# Data Exploration

## Setup

### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [ ]:
import sys
sys.path.append("..")
from my_project.fileManager import FileManager

### Auxiliar functions

In [ ]:
def inspect(df):
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Data types:\n{df.dtypes}")
    print()

### Constants

In [ ]:
RAW_FILE = "../data/raw/automobile_dataset"
PROCESSED_TRAIN_FILE = "../data/processed/automobile_dataset"
PROCESSED_TEST_FILE = "../data/processed/automobile_test"
EXPLORATION_IMAGES_FOLDER = "../reports/figures/exploration/"

## Data loading

We created a FileManager class to make it easier to save and load our dataset using different file formats. The class supports CSV, Parquet, Feather, and Pickle files. The class automatically adds the correct file extension, so we only need to provide the file name. This makes the code more flexible and avoids having to write different functions for each file format.

In [ ]:
fM = FileManager()

In [ ]:
fM.set_format("csv")
df = fM.read(RAW_FILE)

display(df)

Our dataset focuses on analyzing the selling price of used cars. It contains **5500 records** and **18 variables** that describe different aspects of each vehicle: its  specifications, ownership history, condition, and other contextual information. It contains a mixture of numerical and categorical values. The most important variable is the **Selling_Price**, which is the target value which our machine learning models will try to predict as new instances come through. 


In [ ]:
inspect(df)

## Memory optimization

We check how much memory our dataset is using and then try to reduce it by optimizing the data types. First, we calculate the initial memory usage of the DataFrame. Then, we go through all the columns and change their data types to more efficient ones when possible. This helps make the dataset more memory-efficient, which can be useful when working with larger datasets.

In [ ]:
# Initial memory usage
initial_size = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Initial memory usage: {initial_size:.2f} MB")

# Optimize data types
for col in list(df.columns):
    if df.dtypes[col] == "int64":
        df[col] = pd.to_numeric(df[col], downcast='integer')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after downcasting to integer: {size:.2f} MB")
    if df.dtypes[col] == "float64":
        df[col] = pd.to_numeric(df[col], downcast='float')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after downcasting to float: {size:.2f} MB")
    if df.dtypes[col] == "str" or df.dtypes[col] == "object":
        df[col] = df[col].astype('category')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after changing to categorical column: {size:.2f} MB")

# Optimized memory usage
final_size = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Final memory usage: {final_size:.2f} MB")
print(f"Reduction of {(1 - final_size / initial_size) * 100:.2f}%")

In [ ]:
inspect(df)

## Missing values 

Before doing anything, we checked the dataset for missing values using insull().sum(). This allows us to identify in which variables we had these kind of observations. The preprocessing step is important because most machine learning algorithms cannot work directly with missing data.

* The numerical missing values have been replaced  with the mean value of the corresponding column. This is because in the dataset there weren't extreme outliers.
* The categorical variables missing values have been dropped in order to avoid making assumptions about the missing values. 


In [ ]:
print("Missing values:")
print(df.isnull().sum() > 0)
print()

print("Fill missing numerical values:")
dictionary = {}

# Numerical fixes
dfk = df.isnull().sum() > 0
for col in list(df.columns):
    if dfk[col] and df.dtypes[col] != "category":
        dictionary[col] = dfk[col].mean()
df.fillna(dictionary)

# Categorical fixes
print("Drop rows with any missing categorical values:")
print("Categorical Fixes:")
df = df.dropna()
display(df)

# Missing values:
print(df.isnull().sum() > 0)
print()

## Exploration 

#### 1. Car Model Count 

In [ ]:
plt.figure(figsize=(10, 5))
car_counts = df["Make"].value_counts()
plt.bar(car_counts.index, car_counts.values , color='royalblue')
plt.xlabel("Car Model")
plt.ylabel("Count")
plt.xticks(rotation=90) 
plt.title("Automobile Dataset: Car Model Count")
plt.tight_layout()
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}car_model_count.png", dpi=300)

plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
plt.show() # IF WE WANT TO SHOW THE IMAGE

This graph shows the number of cars for each brand in the dataset. We can see that the number of cars is quite similar between the different brands, so there is not a big difference in the representation of each brand. This means that the dataset is relatively balanced in terms of car brands, and it is not heavily dominated by one specific manufacturer

#### 2. Heatmap

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(7, 5))

sns.heatmap( corr, annot=True, cmap='coolwarm' , fmt=".2f")

plt.tight_layout()
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}heatmap.png", dpi=300)

# plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
plt.show() # IF WE WANT TO SHOW THE IMAGE

We can see some high correlations between several variables:
* Horsepower and Torque have a very strong positive correlation (0.98). This means that cars with higher horsepower usually also have higher torque.
* Selling_Price and Year have a positive correlation (0.81). In general, newer cars tend to have a higher selling price.
* Selling_Price has a negative correlation with Mileage (-0.69) and Owners (-0.68). This means that cars with higher mileage or more previous owners tend to have a lower selling price.
* Engine_Size and Fuel_Efficiency have a negative correlation (-0.74). As the engine size increases, fuel efficiency tends to decrease.
* Engine_Size also has a positive correlation with Horsepower (0.67) and Torque (0.56). This suggests that cars with larger engines tend to have more horsepower and torque.

#### 3. Pruebas de Scatterplot

In [ ]:
makes = df["Make"].unique()
n = len(makes)

cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = axes.flatten()

for ax, make in zip(axes, makes):
    df_make = df[df["Make"] == make]

    sns.scatterplot(
        data=df_make,
        x="Selling_Price",
        y="Mileage",
        color="skyblue",
        ax=ax
    )

    ax.set_title(make)

# Eliminar ejes vacíos
for ax in axes[len(makes):]:
    fig.delaxes(ax)

plt.tight_layout()
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}scatterplots.png", dpi=300)

# plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
plt.show() # IF WE WANT TO SHOW THE IMAGE

#### 4. Boxplot by Category

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Make", y="Selling_Price", palette="Set2")
plt.xticks(rotation=90) 
plt.title("Selling Price vs Car ")
plt.tight_layout()

plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}selling_vs_car.png", dpi=300)

# plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
plt.show() # IF WE WANT TO SHOW THE IMAGE

Audi, BMW, and Mercedes-Benz have the highest selling prices in the dataset. These brands also include some of the most expensive vehicles, reflecting their position in the premium car market.

#### 5. Seaborn Pairplot Experiments

In [ ]:
sns.set_theme(style="white")
sns.pairplot(df, hue="Selling_Price", height=2.5)
plt.suptitle("Iris dataset: Pairplot", y=1.02)
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}pairplots.png", dpi=300)

# plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
plt.show() # IF WE WANT TO SHOW THE IMAGE

## Categorical to numerical

In [ ]:
print(df["Make"].nunique())
print(df["Color"].nunique())
print(df["Model"].nunique())
print(df["Service_History"].nunique())
print(df["Body_Type"].nunique())
print(df["Drivetrain"].nunique())
print(df["Location"].nunique())

Machine learning models cannot work directly with categorical data, so these variables must be converted into a numerical format before training. That's why we did some encoding.
* For **Service_History**, we used **ordinal encoding** because the categories have a natural order. A vehicle with no service history provides less information than one with a partial service history.
* For the **remaining categorical variables**, we applied **One-Hot Encoding**. This method creates a separate binary column for each category and avoids introducing relationships between categories that do not actually exist.

In [ ]:
# Ordinal for service (from no service to full service, hierarchy is preserved)
service_mapping = {
    "No Service": 0,
    "Partial Service": 1,
    "Full Service": 2
}

df["Service_History"] = df["Service_History"].map(service_mapping)


# OHE for the other categorical columns
categorical_columns = [
    "Make",
    "Model",
    "Fuel_Type",
    "Transmission",
    "Color",
    "Body_Type",
    "Drivetrain",
    "Location"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    dtype=int
)

In [ ]:
print(df.head())
print(df.dtypes)
print(df.shape)
print(df.columns)

## Save processed data

In [ ]:
from sklearn.model_selection import train_test_split

# 80% training data
# 20% testing data
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=22
)

In [ ]:
from pathlib import Path

Path(PROCESSED_TRAIN_FILE).parent.mkdir(parents=True, exist_ok=True) 

fM.set_format("parquet")
fM.write(train_df, PROCESSED_TRAIN_FILE)
fM.write(test_df, PROCESSED_TEST_FILE)

## Check it was correctly saved

In [ ]:
fM.set_format("parquet")
fdf = fM.read(PROCESSED_TRAIN_FILE)

display(fdf)
inspect(fdf)